In [15]:
# ! 대신 %를 사용합니다. 현재 노트북 커널 전용 환경에 정확히 설치하라는 명령어입니다.
%pip install --upgrade pip
%pip install transformers datasets[audio] accelerate huggingface-hub

Note: you may need to restart the kernel to use updated packages.
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.17-py3-none-any.whl.metadata (6.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached dill-0.4.

ERROR: Could not install packages due to an OSError: [WinError 5] 액세스가 거부되었습니다: 'c:\\Users\\user\\do_it\\venv\\Lib\\site-packages\\psutil\\_psutil_windows.pyd'
Check the permissions.



In [9]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Users\user\ffmpeg\ffmpeg\bin"

In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype = torch_dtype, low_cpu_mem_usage = True, use_safetensors = True
)

model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model = model,
    tokenizer = processor.tokenizer,
    feature_extractor = processor.feature_extractor,
    torch_dtype = torch_dtype,
    device = device,
    return_timestamps = True,
    chunk_length_s = 10,
    stride_length_s = 2,
)

sample = "./audio/lsy_audio_2023_58s.mp3"

result = pipe(sample)

print(result)

ModuleNotFoundError: Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [ ]:
# CSV 파일로 저장하기
start_end_text = []

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns = ["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index = False, sep = "|")
display(df)


NameError: name 'result' is not defined

In [ ]:
to